In [ ]:
import torch
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
model = torch.hub.load('facebookresearch/detr', 'detr_resnet50', pretrained=True).to(device)
model.eval()

In [ ]:
# COCO class names
CLASSES = [
    'N/A', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
    'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat',
    'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
    'bottle', 'N/A', 'wine glass', 'cup', 'fork', 'knife', 'spoon',
    'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli',
    'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch',
    'potted plant', 'bed', 'N/A', 'dining table', 'N/A', 'N/A',
    'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
    'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear',
    'hair drier', 'toothbrush', 'N/A'
]

In [ ]:
print(len(CLASSES))
print(len([label for label in CLASSES if label != 'N/A']))

In [ ]:
!wget -O "twocats.jpg" "http://images.cocodataset.org/val2017/000000039769.jpg"

In [ ]:
# load image
image = Image.open('twocats.jpg').convert("RGB") # 640 x 480 x 3

In [ ]:
# show image
plt.imshow(image)
plt.axis('off')
plt.show()

In [ ]:
image.height, image.width

In [ ]:
# preprocessing
transform = T.Compose([
    T.Resize(800),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])

blob = transform(image).unsqueeze(0)
print(blob.shape)

In [ ]:
# inference
with torch.no_grad():
    outputs = model(blob.to(device))

print(outputs.keys())
print(outputs['pred_logits'].shape)
print(outputs['pred_boxes'].shape)

In [ ]:
# postprocessing
def box_cxcywh_to_xyxy(x):
    x_c, y_c, w, h = x.unbind(1)
    b = [(x_c - 0.5 * w), (y_c - 0.5 * h),
         (x_c + 0.5 * w), (y_c + 0.5 * h)]
    return torch.stack(b, dim=1)

def rescale_bboxes(out_bbox, size):
    img_w, img_h = size
    b = box_cxcywh_to_xyxy(out_bbox)
    b = b * torch.tensor([img_w, img_h, img_w, img_h], dtype=torch.float32)
    return b

prob = outputs['pred_logits'].softmax(-1)[0, :, :-1]
scores, labels = prob.detach().cpu().max(-1)

confidence_threshold = 0.9
keep = scores > confidence_threshold

boxes = rescale_bboxes(outputs['pred_boxes'][0, keep].detach().cpu(), image.size)
labels = labels[keep]
scores = scores[keep]

In [ ]:
torch.set_printoptions(precision=4, sci_mode=False)
print(boxes)
print(labels)
print(scores)

In [ ]:
# visualize result
plt.figure(figsize=(9, 6))
plt.imshow(image)
ax = plt.gca()

for idx, label in enumerate(labels):
    xmin, ymin, xmax, ymax = boxes[idx]
    score = scores[idx]
    ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, color='green', linewidth=3))
    ax.text(xmin, ymin, f'{CLASSES[label]}: {score:.2f}', fontsize=15, bbox=dict(facecolor='yellow', alpha=0.5))

plt.axis('off')
plt.show()

In [ ]:
import sys
sys.path.append('/root/.cache/torch/hub/facebookresearch_detr_main')

In [ ]:
from util.misc import nested_tensor_from_tensor_list

In [ ]:
nested_blob = nested_tensor_from_tensor_list(blob).to(device)

In [ ]:
features, pos = model.backbone(nested_blob)
src, mask = features[-1].decompose()
proj_src = model.input_proj(src)
hs = model.transformer(proj_src, mask, model.query_embed.weight, pos[-1])[0]
outputs_class = model.class_embed(hs)
outputs_coord = model.bbox_embed(hs).sigmoid()
out = {
    'pred_logits': outputs_class[-1],
    'pred_boxes': outputs_coord[-1],
}

In [ ]:
hs.shape # hidden states from all decoder blocks

In [ ]:
features[0].tensors[0].shape, features[0].tensors[0].max().item(), features[0].tensors[0].min().item()

In [ ]:
feats = features[0].tensors[0].detach().cpu().numpy()
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
selected = [0,1,2,3] # select 0-2047
for i, ax in enumerate(axes.flat):
    ax.imshow(feats[selected[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
pos[0].shape, pos[0].max().item(), pos[0].min().item()

In [ ]:
pe = pos[0][0].cpu().numpy()
fig, axes = plt.subplots(16, 16, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(pe[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
mask.shape, mask.max().item(), mask.min().item()

In [ ]:
plt.imshow(mask[0].cpu().numpy())
plt.axis('off')
plt.show()

In [ ]:
proj_src.shape, proj_src.max().item(), proj_src.min().item()

In [ ]:
feats = proj_src[0].detach().cpu().numpy()
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
selected = [0,1,2,3] # select 0-255
for i, ax in enumerate(axes.flat):
    ax.imshow(feats[selected[i]])
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
hs.shape, hs.max().item(), hs.min().item()

In [ ]:
# head run on each decoder block
outputs_class = model.class_embed(hs)
outputs_coord = model.bbox_embed(hs).sigmoid()
outputs = {
    'pred_logits': outputs_class,
    'pred_boxes': outputs_coord,
}

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
axes = axes.flatten()

for i in range(len(hs)):
    # postprocessing
    prob = outputs['pred_logits'][i].softmax(-1)[0, :, :-1]
    scores, labels = prob.detach().cpu().max(-1)
    confidence_threshold = 0.8
    keep = scores > confidence_threshold
    boxes = rescale_bboxes(outputs['pred_boxes'][i][0, keep].detach().cpu(), image.size)
    labels = labels[keep]
    scores = scores[keep]
    # visualization
    ax = axes[i]
    ax.imshow(image)
    for idx, label in enumerate(labels):
        xmin, ymin, xmax, ymax = boxes[idx]
        score = scores[idx]
        ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, color='green', linewidth=2))
        ax.text(xmin, ymin, f'{CLASSES[label]}: {score:.2f}', fontsize=10, bbox=dict(facecolor='yellow', alpha=0.5))
    ax.set_title(f"Decoder layer {i+1}")
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import inspect

In [ ]:
print(inspect.getsource(model.transformer.forward))

In [ ]:
print(inspect.getsource(model.transformer.encoder.forward))

In [ ]:
print(inspect.getsource(model.transformer.decoder.forward))

In [ ]:
model.query_embed.weight.shape # pe

In [ ]:
hs0 = torch.zeros_like(model.query_embed.weight.unsqueeze(1).repeat(1, 1, 1)).permute(1,0,2).unsqueeze(0) # starting hidden state
print(hs0.shape)

In [ ]:
# head
outputs_class = model.class_embed(hs0)
outputs_coord = model.bbox_embed(hs0).sigmoid()
outputs = {
    'pred_logits': outputs_class[0],
    'pred_boxes': outputs_coord[0],
}

In [ ]:
# postprocessing
prob = outputs['pred_logits'].softmax(-1)[0, :, :-1]
scores, labels = prob.detach().cpu().max(-1)
confidence_threshold = 0.01345
keep = scores > confidence_threshold
boxes = rescale_bboxes(outputs['pred_boxes'][0, keep].detach().cpu(), image.size)
labels = labels[keep]
scores = scores[keep]
print(len(labels))

In [ ]:
# visualize initial hidden state
plt.figure(figsize=(4.5, 3))
plt.imshow(image)
ax = plt.gca()

for idx, label in enumerate(labels):
    xmin, ymin, xmax, ymax = boxes[idx]
    score = scores[idx]
    ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, color='green', linewidth=3))
    ax.text(xmin, ymin, f'{CLASSES[label]}: {score:.2f}', fontsize=15, bbox=dict(facecolor='yellow', alpha=0.5))

plt.axis('off')
plt.show()

In [ ]:
hs0 = model.query_embed.weight.unsqueeze(1).repeat(1, 1, 1).permute(1,0,2).unsqueeze(0) # starting hidden state + pe
print(hs0.shape)

In [ ]:
# head
outputs_class = model.class_embed(hs0)
outputs_coord = model.bbox_embed(hs0).sigmoid()
outputs = {
    'pred_logits': outputs_class[0],
    'pred_boxes': outputs_coord[0],
}

In [ ]:
# postprocessing
prob = outputs['pred_logits'].softmax(-1)[0, :, :-1]
scores, labels = prob.detach().cpu().max(-1)
confidence_threshold = 0.8
keep = scores > confidence_threshold
boxes = rescale_bboxes(outputs['pred_boxes'][0, keep].detach().cpu(), image.size)
labels = labels[keep]
scores = scores[keep]
print(len(labels))

In [ ]:
# visualize initial hidden state + pe
plt.figure(figsize=(4.5, 3))
plt.imshow(image)
ax = plt.gca()

for idx, label in enumerate(labels):
    xmin, ymin, xmax, ymax = boxes[idx]
    score = scores[idx]
    ax.add_patch(plt.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, fill=False, color='green', linewidth=3))
    ax.text(xmin, ymin, f'{CLASSES[label]}: {score:.2f}', fontsize=15, bbox=dict(facecolor='yellow', alpha=0.5))

plt.axis('off')
plt.show()